# Tarea 4 - Reconocimiento de Patrones en Imágenes

**Laura Romero 🇮🇹 y Javier Oberto** 👅

---

### Instrucciones de uso
1. crear entorno virtual de python
    *python -m venv 'nombre del env'*

    *source 'nombre del env'/bin/activate*

2. Colocar la carpeta `BD/` en el mismo directorio que este notebook.
3. Ejecutar las celdas en orden de arriba hacia abajo.
4. La celda de SBS puede tardar en ejecutar, a la calma nomas.
5. paso 5: disfrutar

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install pandas
!pip install scipy
!pip install scikit-learn
!pip install scikit-image

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
from sklearn import preprocessing
from math import radians
import os
import glob

---
## Parte I: Construcción de la Base de Datos
Construimos la base de datos donde extraemos 6 descriptores de la matriz de co-ocurrencia (GLCM) por cada canal (R, G, B y escala de grises), lo que nos da 24 características por imagen. Usamos distancia=1, ángulo=90°, 16 niveles de gris y `symmetric=True` como pide el enunciado.

In [ ]:
# propiedades a extraer
features_glcm = ['contrast', 'correlation', 'dissimilarity', 'homogeneity', 'ASM', 'energy']

canales = ['R', 'G', 'B', 'gray']

nombres_columnas = []
for canal in canales:
    for feat in features_glcm:
        nombres_columnas.append(f'{canal}_{feat}')

print(nombres_columnas)

In [ ]:
# leemos las carpetas de texturas
carpetas = sorted(glob.glob('BD/textura_*'))

# aqui guardamos todos los descriptores y las clases
F = []
etiquetas = []

for carpeta in carpetas:

    clase = int(carpeta.split('_')[-1])
    archivos = sorted(glob.glob(os.path.join(carpeta, '*.jpg')))

    for archivo in archivos:

        img = cv2.imread(archivo)

        blue, green, red = cv2.split(img)

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        descriptores_imagen = []

        for canal in [red, green, blue, gray]:

            canal_col = canal.reshape(-1, 1)
            canal_escalado = preprocessing.MinMaxScaler((0, 15)).fit_transform(canal_col).astype(int)
            canal_escalado = canal_escalado.reshape(canal.shape)

            niveles = np.max(canal_escalado) + 1
            # symmetric=True como pide el enunciado de la tarea 4
            glcm = graycomatrix(canal_escalado, distances=[1], angles=[radians(90)],
                                levels=niveles, symmetric=True, normed=False)

            for feat in features_glcm:
                valor = graycoprops(glcm, feat).squeeze()
                descriptores_imagen.append(float(valor))

        F.append(descriptores_imagen)
        etiquetas.append(clase)

# armamos el dataframe
df = pd.DataFrame(F, columns=nombres_columnas)
df['clase'] = etiquetas

print('el dataset quedo de', df.shape)
print(df.head(10))

---
## Parte II: Normalización
Normalizamos cada columna con media cero y desviación uno usando `StandardScaler` igual que en clases.

In [ ]:
from sklearn.preprocessing import StandardScaler

# separamos features y etiquetas
X = df[nombres_columnas].values
y = df['clase'].values

# normalizamos: media=0, desviacion=1
escala = StandardScaler()
X_norm = escala.fit_transform(X)

print('medias:', np.round(X_norm.mean(axis=0), 4))
print()
print('desviaciones:', np.round(X_norm.std(axis=0), 4))

# lo guardamos en un dataframe para tenerlo organizado
df_norm = pd.DataFrame(X_norm, columns=nombres_columnas)
df_norm['clase'] = y
print('\ndataset normalizado:', df_norm.shape)

---
## Parte III: Selección de Características (SBS + Fisher)
Usamos la Sequential Backward Selection: partiendo de 24 características vamos eliminando la peor de a una, quedándonos con la combinación que maximiza el índice de Fisher en cada paso.

La función de Fisher viene directo de lo visto en la clase 15.

In [ ]:
from numpy.linalg import inv
from numpy.matlib import repmat
from itertools import combinations

# funcion que calcula el indice de Fisher (clase 15)
def fisher_extraction_list(data, cols, clases):
    if cols == 1:
        D = np.hstack(data)
        total = len(D)
        Vm = np.mean(D)
    else:
        D = np.vstack(data)
        Vm = np.mean(D, axis=0)
        total = D.shape[0]

    p = np.zeros((clases, 1))
    Vk = np.zeros((clases, cols))
    Gn = []

    for i in range(clases):
        Vk[i, :] = np.mean(data[i], axis=0)
        pts = data[i].shape[0]
        if cols == 1:
            Gn.append(data[i] - Vm)
        else:
            Gn.append(data[i] - repmat(Vm, pts, 1))
        p[i] = data[i].shape[0] / total

    Cb = np.zeros((cols, cols))
    Cw = np.zeros((cols, cols))

    for k in range(clases):
        Cb = Cb + p[k] * np.matmul((Vk[k, :] - Vm).reshape(cols, 1),
                                    (Vk[k, :] - Vm).reshape(1, cols))
        MGn = np.array(Gn[k])
        Cw = Cw + p[k] * np.cov(MGn.T)

    J = np.trace(np.matmul(inv(Cw), Cb))
    return J

In [ ]:
# funcion SBS (clase 15)

def sbs_procedure(X_data, clases, feature_names, sel_features):
    cols = X_data.shape[1]
    id_clases = np.unique(clases)
    u_clases = len(id_clases)

    indices_actuales = np.arange(cols)
    historial_J = []
    historial_features = []

    t = cols
    while t > sel_features:
        combs = np.array(list(combinations(indices_actuales, len(indices_actuales) - 1)))
        J = np.zeros(len(combs))

        for i in range(len(combs)):
            feat = combs[i]
            sub_data = []
            for k in id_clases:
                id_class = clases == k
                sub_data.append(X_data[id_class][:, feat])

            J[i] = fisher_extraction_list(sub_data, len(feat), u_clases)

        best = np.argmax(J)
        indices_actuales = combs[best]
        best_J = J[best]

        historial_J.append(best_J)
        historial_features.append(feature_names[indices_actuales].copy())

        print(f'  quedan {len(indices_actuales)} features, Fisher = {best_J:.4f}')
        t = t - 1

    return indices_actuales, historial_J, historial_features

In [ ]:
# corremos SBS desde 24 features hasta 3 (para evaluar cual maximiza fisher)
feature_names = np.array(nombres_columnas)

print('corriendo SBS, esto tarda un poco \n')
indices_sel, historial_J, historial_features = sbs_procedure(X_norm, y, feature_names, 3)

print('\nal final quedaron estas 3:', list(feature_names[indices_sel]))

In [ ]:
# graficamos Fisher vs numero de caracteristicas para ver la tendencia
n_features_list = list(range(23, 2, -1))

plt.figure(figsize=(10, 5))
plt.plot(n_features_list, historial_J, 'x-', color='purple')
plt.xlabel('Número de características')
plt.ylabel('Índice de Fisher')
plt.title('SBS: Índice de Fisher vs Número de Características')
plt.grid(alpha=0.3)
plt.gca().invert_xaxis()
plt.show()

### Probamos distintas cantidades de características
Evaluamos los 3 clasificadores que pide la tarea con distintos numeros de features para encontrar el minimo que maximice el rendimiento.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

# split 70/30 estratificado (misma funcion de la tarea 3)
def split_train_test(X, y, porc_train=0.7, seed=42):
    rng = np.random.default_rng(seed)
    idx_train = []
    idx_test = []
    for clase in np.unique(y):
        idx_clase = np.where(y == clase)[0]
        rng.shuffle(idx_clase)
        corte = int(len(idx_clase) * porc_train)
        idx_train.extend(idx_clase[:corte])
        idx_test.extend(idx_clase[corte:])
    return np.array(idx_train), np.array(idx_test)

idx_train, idx_test = split_train_test(X_norm, y)
print('muestras de entrenamiento:', len(idx_train))
print('muestras de prueba:', len(idx_test))

In [ ]:
# probamos con 5 valores distintos de features
valores_a_probar = [20, 15, 10, 7, 5]

# los 3 clasificadores que pide la tarea
clasificadores = {
    'Naive Bayes': GaussianNB(),
    'Arbol de Decision': DecisionTreeClassifier(random_state=42),
    'KNN (K=5)': KNeighborsClassifier(n_neighbors=5)
}

# guardamos resultados para cada clasificador y cada cantidad de features
resultados = {nombre: [] for nombre in clasificadores}

for n in valores_a_probar:
    idx = 23 - n
    feats = historial_features[idx]
    cols = [nombres_columnas.index(f) for f in feats]

    # usamos datos sin normalizar y normalizamos solo con train (sin data leakage)
    X_n = X[:, cols]
    X_train_raw, y_train = X_n[idx_train], y[idx_train]
    X_test_raw, y_test = X_n[idx_test], y[idx_test]

    escala_cv = StandardScaler()
    X_train = escala_cv.fit_transform(X_train_raw)
    X_test = escala_cv.transform(X_test_raw)

    print(f'\n--- {n} features ---')
    for nombre, clf in clasificadores.items():
        # clonamos el clasificador para que no se contamine entre iteraciones
        from sklearn.base import clone
        modelo = clone(clf)
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        # f1 promedio weighted porque son 10 clases
        fscore = f1_score(y_test, y_pred, average='weighted')
        resultados[nombre].append(fscore)
        print(f'  {nombre}: F-score = {fscore:.4f}')

In [ ]:
# graficamos f-score vs numero de caracteristicas para los 3 clasificadores
plt.figure(figsize=(10, 5))
colores_clf = ['blue', 'green', 'orange']
for i, (nombre, scores) in enumerate(resultados.items()):
    plt.plot(valores_a_probar, scores, 'o-', color=colores_clf[i], label=nombre)

plt.xlabel('Número de características')
plt.ylabel('F-score')
plt.title('F-score vs Número de Características por Clasificador')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Elegimos el numero de caracteristicas
Buscamos el minimo numero de features que maximice el rendimiento promedio de los clasificadores.

In [ ]:
# promedio de f-score de los 3 clasificadores para cada cantidad de features
promedios = []
for i, n in enumerate(valores_a_probar):
    promedio = np.mean([resultados[nombre][i] for nombre in clasificadores])
    promedios.append(promedio)
    print(f'{n} features -> F-score promedio = {promedio:.4f}')

# elegimos el minimo numero de features donde el f-score no baja mucho
mejor_idx = np.argmax(promedios)
n_elegido = valores_a_probar[mejor_idx]
print(f'\nmejor cantidad: {n_elegido} features con F-score promedio = {promedios[mejor_idx]:.4f}')

In [ ]:
# nos quedamos con las features elegidas
idx_elegido = 23 - n_elegido
mejor_features = historial_features[idx_elegido]
indices_mejor = [nombres_columnas.index(f) for f in mejor_features]

print(f'elegimos {n_elegido} caracteristicas')
print('Fisher:', round(historial_J[idx_elegido], 4))
print('features:', list(mejor_features))

---
## Parte IV: Clasificadores
Comparamos 3 clasificadores supervisados con split 70% entrenamiento y 30% test:
1. **Naive Bayes** (GaussianNB) - visto en la clase 19
2. **Árbol de decisión** (DecisionTreeClassifier)
3. **KNN** con K=5 - K nearest neighbors

Normalizamos solo con los datos de train para evitar data leakage (igual que en la tarea 3).

In [ ]:
# preparamos los datos con las features elegidas
X_sel = X[:, indices_mejor]

X_train_raw, y_train = X_sel[idx_train], y[idx_train]
X_test_raw, y_test = X_sel[idx_test], y[idx_test]

# normalizamos con train solamente
escala_final = StandardScaler()
X_train = escala_final.fit_transform(X_train_raw)
X_test = escala_final.transform(X_test_raw)

print('train:', X_train.shape, '| test:', X_test.shape)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# entrenamos y evaluamos los 3 clasificadores
modelos_entrenados = {}
f_scores = {}

for nombre, clf in clasificadores.items():
    from sklearn.base import clone
    modelo = clone(clf)
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    fscore = f1_score(y_test, y_pred, average='weighted')
    f_scores[nombre] = fscore
    modelos_entrenados[nombre] = modelo

    print(f'\n{"="*50}')
    print(f'{nombre} - F-score: {fscore:.4f}')
    print(f'{"="*50}')
    print(classification_report(y_test, y_pred))

---
## Parte V: Rendimiento
Graficamos el F-score de cada clasificador y determinamos cuál es el mejor. Además comparamos el rendimiento **con y sin normalización**.

In [ ]:
# grafico de barras comparando f-score de los 3 clasificadores
nombres = list(f_scores.keys())
scores = list(f_scores.values())

plt.figure(figsize=(8, 5))
barras = plt.bar(nombres, scores, color=['blue', 'green', 'orange'], alpha=0.8)

# ponemos el valor encima de cada barra
for barra, score in zip(barras, scores):
    plt.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.005,
             f'{score:.4f}', ha='center', fontsize=11)

plt.ylabel('F-score')
plt.title('Comparación de Clasificadores (con normalización)')
plt.ylim(0, 1.1)
plt.grid(axis='y', alpha=0.3)
plt.show()

mejor = max(f_scores, key=f_scores.get)
print(f'el mejor clasificador es: {mejor} con F-score = {f_scores[mejor]:.4f}')

### Comparación con y sin normalización
Ahora repetimos todo pero **sin normalizar** los datos para ver si cambia el rendimiento. Esto es para verificar el efecto del StandardScaler en cada clasificador.

In [ ]:
# evaluamos los mismos 3 clasificadores pero SIN normalizar
f_scores_sin_norm = {}

for nombre, clf in clasificadores.items():
    from sklearn.base import clone
    modelo = clone(clf)
    # usamos los datos crudos, sin StandardScaler
    modelo.fit(X_train_raw, y_train)
    y_pred = modelo.predict(X_test_raw)

    fscore = f1_score(y_test, y_pred, average='weighted')
    f_scores_sin_norm[nombre] = fscore
    print(f'{nombre} SIN normalizar: F-score = {fscore:.4f}')

In [ ]:
# grafico comparativo: con normalizacion vs sin normalizacion
x_pos = np.arange(len(nombres))
ancho = 0.35

plt.figure(figsize=(10, 5))
barras1 = plt.bar(x_pos - ancho/2, [f_scores[n] for n in nombres], ancho,
                   label='Con normalización', color='steelblue', alpha=0.8)
barras2 = plt.bar(x_pos + ancho/2, [f_scores_sin_norm[n] for n in nombres], ancho,
                   label='Sin normalización', color='coral', alpha=0.8)

# valores encima de las barras
for barra in barras1:
    plt.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.005,
             f'{barra.get_height():.3f}', ha='center', fontsize=9)
for barra in barras2:
    plt.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.005,
             f'{barra.get_height():.3f}', ha='center', fontsize=9)

plt.xticks(x_pos, nombres)
plt.ylabel('F-score')
plt.title('Rendimiento Con vs Sin Normalización')
plt.legend()
plt.ylim(0, 1.15)
plt.grid(axis='y', alpha=0.3)
plt.show()

### Tabla resumen

In [ ]:
# tabla resumen final
print(f'{"Clasificador":<22} {"Con Norm":<12} {"Sin Norm":<12} {"Diferencia":<12}')
print('-' * 55)
for nombre in nombres:
    con = f_scores[nombre]
    sin = f_scores_sin_norm[nombre]
    diff = con - sin
    print(f'{nombre:<22} {con:<12.4f} {sin:<12.4f} {diff:<+12.4f}')

### Análisis de resultados

**Efecto de la normalización:**
- **KNN** es el clasificador más afectado por la normalización ya que calcula distancias entre puntos. Sin normalizar, las features con valores más grandes dominan la distancia y las de valores chicos no aportan nada. Al normalizar todas quedan en la misma escala y KNN puede compararlas de forma justa.
- **Naive Bayes** asume que cada feature sigue una distribución gaussiana y calcula la probabilidad de forma independiente por cada una, por lo que la escala no le afecta tanto porque cada feature tiene su propia media y varianza.
- **Árbol de decisión** no se ve afectado por la normalización porque trabaja con umbrales de corte sobre cada feature individual. Le da igual si los valores van de 0 a 1 o de 0 a 1000, el corte lo encuentra igual.

En resumen, normalizar es clave para clasificadores basados en distancia (KNN), pero no cambia el rendimiento de los basados en reglas o probabilidad independiente.